# 0. ENVIRONMENT SETUP

In [ ]:
import os
print("CPU:", os.cpu_count())

CPU: 2


In [ ]:
!nvidia-smi || echo "No GPU detected"
!pip install -q sentence-transformers faiss-cpu nltk pandas beautifulsoup4
!pip install --upgrade -q sentence-transformers

Thu Sep 24 01:33:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 1. LOAD DATASET

In [ ]:
import faiss, torch, nltk, pickle, re, string
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from bs4 import BeautifulSoup
from collections import defaultdict

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
GITHUB_RELEASE_URL = (
    "https://github.com/briyanyehezkhiel/github-issues-dataset/"
    "releases/download/1.0/github_issues_tickets.csv"
)

df = pd.read_csv(GITHUB_RELEASE_URL)
print("Dataset shape:", df.shape)
df.head(2)

Dataset shape: (15955, 132)


/tmp/ipykernel_693/4156777671.py:6: DtypeWarning: Columns (43,48,49,52,53,104) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(GITHUB_RELEASE_URL)


,answers_0_author,answers_0_body,answers_0_creation_time,answers_1_author,answers_1_body,answers_1_creation_time,answers_2_author,answers_2_body,answers_2_creation_time,answers_3_author,...,repo_name,state,state_reason,title,updated_at,url,user_id,user_login,user_type,user_url
0,MukulKolpe,"Hey @alan-agius4, can I work on this issue?",2023-05-05T18:09:31+00:00,JeanMeche,@MukulKolpe You can open a PR if you'd like to...,2023-05-05T18:19:32+00:00,jamesanderson9182,"Following the history and looking at the page,...",2023-05-16T21:51:45+00:00,angular-automatic-lock-bot[bot],...,angular/angular,closed,completed,Article mistake,2023-06-19T00:09:05Z,https://api.github.com/repos/angular/angular/i...,96018508,PiotrZinfWSB,User,https://api.github.com/users/PiotrZinfWSB
1,github-actions[bot],Hi I'm an AI powered bot that finds similar is...,2024-06-07T20:52:50+00:00,codendone,"This is very similar to #9670, though that iss...",2024-06-08T02:22:56+00:00,codendone,This was closed as fixed/duplicate of #7629. T...,2024-07-11T02:02:16+00:00,NaN,...,microsoft/microsoft-ui-xaml,closed,completed,Able to change the window height even if IsRes...,2024-08-14T18:11:31Z,https://api.github.com/repos/microsoft/microso...,50815710,eduardobragaxz,User,https://api.github.com/users/eduardobragaxz


# 2. DATA FILTERING & LABEL PREPARATION

## 2.1 Filter Closed Issues

In [ ]:
df = df[
    (df["state"] == "closed") &
    (df["closed_at"].notnull())
]


## 2.2 Filter Technical Labels

In [ ]:
TARGET_LABELS = [
    "bug", "error", "exception",
    "question", "technical", "help wanted"
]

def has_target_label(row):
    for i in range(11):
        col = f"labels_{i}_name"
        if col in row and pd.notnull(row[col]):
            if row[col].lower() in TARGET_LABELS:
                return True
    return False

df = df[df.apply(has_target_label, axis=1)]
print("After label filter:", df.shape)


After label filter: (13665, 132)


## 2.3 Extract Primary Labels

In [ ]:
def extract_primary_label(row):
    for i in range(11):
        col = f"labels_{i}_name"
        if col in row and pd.notnull(row[col]):
            return row[col].lower()
    return "other"

df["primary_label"] = df.apply(extract_primary_label, axis=1)

print(df["primary_label"].value_counts().head())


primary_label
bug                     12865
help wanted               155
no code attached yet       57
enhancement                45
question                   31
Name: count, dtype: int64


## 2.4 Build Question–Answer Pairs

In [ ]:
df["question"] = df["title"].fillna("") + " " + df["body"].fillna("")


In [ ]:
def extract_best_answer(row):
    for i in range(10):
        col = f"answers_{i}_body"
        if col in row and pd.notnull(row[col]) and len(row[col]) > 50:
            return row[col]
    return None

df["answer"] = df.apply(extract_best_answer, axis=1)
df = df[df["answer"].notnull()]
print("After answer filter:", df.shape)


After answer filter: (13055, 135)


# 3. Text Preprocessing

In [ ]:
def clean_html(text):
    return BeautifulSoup(text, "html.parser").get_text()

def truncate_code(text):
    text = re.sub(r"```[\s\S]*?```", " <CODE> ", text)
    text = re.sub(r"`[^`]+`", " <CODE> ", text)
    return text

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunk_text(text, max_words=150):
    sentences = nltk.sent_tokenize(text)
    chunks, current, count = [], [], 0

    for s in sentences:
        words = s.split()
        if count + len(words) > max_words:
            chunks.append(" ".join(current))
            current, count = [], 0
        current.append(s)
        count += len(words)

    if current:
        chunks.append(" ".join(current))
    return chunks


In [ ]:
def preprocess_for_embedding(text):
    text = clean_html(text)
    text = truncate_code(text)
    text = normalize_text(text)
    return chunk_text(text)


# 4. Build Retrieval Corpus

In [ ]:
corpus = []
metadata = []

print("Building inference corpus...")

for _, row in df.iterrows():
    # ===== ISSUE BODY =====
    body_text = f"{row['title']} {row['body']}"
    body_text = body_text[:1500]

    corpus.append(body_text)
    metadata.append({
        "issue_id": row["id"],
        "title": row["title"],
        "url": row["html_url"],
        "repo": row["repo_name"],
        "type": "issue_body",
        "label": row["primary_label"],

        # reaction metadata
        "reactions_plus_1": row.get("reactions_plus_1", 0),
        "reactions_heart": row.get("reactions_heart", 0),
        "reactions_rocket": row.get("reactions_rocket", 0),
        "reactions_hooray": row.get("reactions_hooray", 0),
        "reactions_minus_1": row.get("reactions_minus_1", 0)
    })

    # ===== ANSWER CHUNKS =====
    chunks = preprocess_for_embedding(row["answer"])
    for c in chunks:
        if len(c.split()) > 5:
            corpus.append(f"{row['title']}. Solution: {c}")
            metadata.append({
                "issue_id": row["id"],
                "title": row["title"],
                "url": row["html_url"],
                "repo": row["repo_name"],
                "type": "issue_answer",
                "label": row["primary_label"],

                # reaction metadata
                "reactions_plus_1": row.get("reactions_plus_1", 0),
                "reactions_heart": row.get("reactions_heart", 0),
                "reactions_rocket": row.get("reactions_rocket", 0),
                "reactions_hooray": row.get("reactions_hooray", 0),
                "reactions_minus_1": row.get("reactions_minus_1", 0)
            })

print("Total vectors:", len(corpus))

Building inference corpus...


/tmp/ipykernel_693/4138378036.py:2: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  return BeautifulSoup(text, "html.parser").get_text()


Total vectors: 26077


# 6. LOAD Deployment Artifact

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import faiss
import pickle
import torch
import nltk
import re
import numpy as np

from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import T5Tokenizer, T5ForConditionalGeneration

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/ILKOM_LOVERS/Skripsi/Skripsi_2/rag_sbert_project_backup"

# ======================================
# LOAD SBERT MODEL
# ======================================

model = SentenceTransformer(
    f"{PROJECT_DIR}/sbert_epoch_1",
    #"all-MiniLM-L6-V2",
    device="cuda"
)

print("SBERT loaded!")

# ======================================
# LOAD FAISS
# ======================================

index = faiss.read_index(
    f"{PROJECT_DIR}/best_faiss.index"
)

print("FAISS loaded!")

# ======================================
# LOAD CORPUS
# ======================================

with open(f"{PROJECT_DIR}/corpus.pkl", "rb") as f:
    corpus = pickle.load(f)

# ======================================
# LOAD METADATA
# ======================================

with open(f"{PROJECT_DIR}/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print("Corpus & metadata loaded!")
print("Total vectors:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SBERT loaded!
FAISS loaded!
Corpus & metadata loaded!
Total vectors: 26077


# 8. DENSE RETRIEVAL PIPELINE

## 8.1 Encode Query

In [ ]:
def encode_query(query):
    query = normalize_text(query)
    return model.encode(
        query,
        prompt_name="query",
        convert_to_numpy=True,
        normalize_embeddings=True
    )


## 8.2 Retrieve Top Candidates

In [ ]:
def retrieve(query, k=5):
    q_emb = encode_query(query)
    scores, idxs = index.search(np.array([q_emb]), k*3)

    candidates = []
    for score, idx in zip(scores[0], idxs[0]):
        meta = metadata[idx]

        # === RERANK RULES ===
        final_score = score
        if meta["type"] == "issue_answer":
            final_score *= 1.2
        final_score *= (1 + np.log1p(meta["likes"]))

        candidates.append((final_score, idx))

    candidates.sort(reverse=True)
    top = candidates[:k]

    results = []
    for rank, (_, idx) in enumerate(top, 1):
        results.append({
            "rank": rank,
            "text": corpus[idx],
            "issue_id": metadata[idx]["issue_id"],
            "url": metadata[idx]["url"],
            "type": metadata[idx]["type"]
        })
    return results


## 8.3 Metadata Score Boosting

In [ ]:
def retrieve_indices(query, k=10):
    emb = encode_query(query)
    scores, idxs = index.search(np.array([emb]), k)
    return idxs[0]


# 9. CROSS-ENCODER RERANKING

## 9.1 Load Reranker

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda"
)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## 9.2 Reaction Boost

Reaction yang kita pakai:

👍 plus_1

❤️ heart

🚀 rocket

🎉 hooray

👎 minus_1 (penalty)

In [ ]:
#import math

#def reaction_boost(meta):
#    score = 1.0
#    score += math.log1p(meta.get("reactions_plus_1", 0))
#    score += 0.5 * math.log1p(meta.get("reactions_heart", 0))
#    score += 0.3 * math.log1p(meta.get("reactions_rocket", 0))
#    score += 0.3 * math.log1p(meta.get("reactions_hooray", 0))
#    score -= 0.7 * math.log1p(meta.get("reactions_minus_1", 0))
#    return max(score, 0.1)


In [ ]:
def reaction_boost(meta):
    return 1.0

## 9.3 Final Reranking

In [ ]:
def rerank(query, candidate_idxs, top_k=10):
    pairs = []

    for idx in candidate_idxs:
        text = corpus[idx]
        if metadata[idx]["type"] == "issue_body":
            text = text[:250]
        pairs.append((query, text))

    ce_scores = reranker.predict(pairs)

    scored = []
    for idx, ce_score in zip(candidate_idxs, ce_scores):
        boost = reaction_boost(metadata[idx])
        final_score = ce_score * boost
        scored.append((idx, final_score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in scored[:top_k]]


In [ ]:
def retrieve_sbert(query, k=100):
    emb = encode_query(query)
    _, idxs = index.search(np.array([emb]), k)
    return idxs[0]

In [ ]:
def retrieve_dense_reranked(query, k=10):
    candidates = retrieve_sbert(query, k=80)
    return rerank(query, candidates, top_k=k)

In [ ]:
def retrieve_indices(query, k=10):
    return retrieve_dense_reranked(query, k=k)

# 10. RETRIEVAL EVALUATION

## 10.1 Basic Retrieval Metrics

### Accuracy@1

In [ ]:
def evaluate_accuracy_at_1(df, max_samples=300):
    correct = 0
    total = 0

    sampled_df = df.sample(min(len(df), max_samples), random_state=42)

    for _, row in sampled_df.iterrows():
        query = str(row["title"]).strip()
        if not query:
            continue

        retrieved = retrieve_indices(query, k=1)
        if len(retrieved) == 0:
            continue

        total += 1
        retrieved_issue_id = metadata[retrieved[0]]["issue_id"]

        if retrieved_issue_id == row["id"]:
            correct += 1

    return correct / total if total > 0 else 0.0


### MRR@10

In [ ]:
def evaluate_mrr(df, k=10, max_samples=300):
    reciprocal_ranks = []

    sampled_df = df.sample(min(len(df), max_samples), random_state=42)

    for _, row in sampled_df.iterrows():
        query = str(row["title"]).strip()
        if not query:
            continue

        retrieved = retrieve_indices(query, k=k)

        rank = 0
        for i, idx in enumerate(retrieved, start=1):
            if metadata[idx]["issue_id"] == row["id"]:
                rank = i
                break

        if rank > 0:
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

    return np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0


### nDCG@5

In [ ]:
import math

def dcg_at_k(relevance, k):
    return sum(
        rel / math.log2(i + 2)
        for i, rel in enumerate(relevance[:k])
    )

def ndcg_at_k(relevance, k):
    ideal = sorted(relevance, reverse=True)
    ideal_dcg = dcg_at_k(ideal, k)
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(relevance, k) / ideal_dcg


In [ ]:
def evaluate_ndcg(df, k=5, max_samples=300):
    scores = []

    sampled_df = df.sample(min(len(df), max_samples), random_state=42)

    for _, row in sampled_df.iterrows():
        query = str(row["title"]).strip()
        if not query:
            continue

        retrieved = retrieve_indices(query, k=k)

        relevance = [
            1 if metadata[idx]["issue_id"] == row["id"] else 0
            for idx in retrieved
        ]

        scores.append(ndcg_at_k(relevance, k))

    return np.mean(scores) if scores else 0.0


### Evaluasi

In [ ]:
#acc = evaluate_accuracy_at_1(df)
#mrr = evaluate_mrr(df, k=10)
#ndcg = evaluate_ndcg(df, k=5)#

#print(f"Accuracy@1 : {acc:.4f}")
#print(f"MRR@10     : {mrr:.4f}")
#print(f"nDCG@5     : {ndcg:.4f}")


## 10.2 Gold Query Dataset

In [ ]:
#gold_candidates = df.sample(50, random_state=42)[
#    ["id", "title", "body", "repo_name"]
#].reset_index(drop=True)

#gold_candidates


In [ ]:
#gold_template = gold_candidates[["id"]].copy()
#gold_template["gold_query"] = ""

#gold_template.to_csv("gold_queries_template.csv", index=False)
#print("Template saved as gold_queries_template.csv")


In [ ]:
#gold_queries = []

#print("=== GOLD QUERY ENTRY MODE ===")
#print("Tulis query user nyata. Ketik 'skip' untuk lewati issue.\n")

#for i, row in gold_candidates.iterrows():
#    print("="*60)
#    print(f"Issue {i+1}/50")
#    print("Repo :", row["repo_name"])
#    print("ID   :", row["id"])
#    print("Title:", row["title"])
#    print("Body :", str(row["body"])[:300], "...\n")

#    q = input("👉 Gold Query (user style): ").strip()

#    if q.lower() == "skip" or q == "":
#        print("⏭️  Skipped")
#        continue

#    gold_queries.append({
#        "issue_id": row["id"],
#        "gold_query": q
#    })

#print("\nTotal gold queries collected:", len(gold_queries))


In [ ]:
#gold_df = pd.DataFrame(gold_queries)
#gold_df.to_csv("gold_queries.csv", index=False)
#print("Saved as gold_queries.csv")


In [ ]:
#import pandas as pd

#gold_df = pd.read_csv("gold_queries.csv")

#print("Jumlah gold query:", len(gold_df))
#gold_df.head()


In [ ]:
#valid_ids = set(m["issue_id"] for m in metadata)
#missing = [i for i in gold_df["issue_id"] if i not in valid_ids]

#print("Missing issue_id:", missing)


In [ ]:
#def retrieve_indices(query, k=10):
#    emb = encode_query(query)
#    _, idxs = index.search(np.array([emb]), k)
#    return idxs[0]


## 10.3 Gold Query Evaluation

### Accuracy@1

In [ ]:
def evaluate_accuracy_gold(gold_df):
    correct = 0

    for _, row in gold_df.iterrows():
        query = row["gold_query"]
        issue_id = row["issue_id"]

        retrieved = retrieve_indices(query, k=1)
        if len(retrieved) == 0:
            continue

        if metadata[retrieved[0]]["issue_id"] == issue_id:
            correct += 1

    return correct / len(gold_df)


### MRR@10

In [ ]:
#import numpy as np

def evaluate_mrr_gold(gold_df, k=10):
    scores = []

    for _, row in gold_df.iterrows():
        query = row["gold_query"]
        issue_id = row["issue_id"]

        retrieved = retrieve_indices(query, k=k)

        rank = 0
        for i, idx in enumerate(retrieved, start=1):
            if metadata[idx]["issue_id"] == issue_id:
                rank = i
                break

        scores.append(1 / rank if rank > 0 else 0)

    return np.mean(scores)


### nDCG@5

In [ ]:
import math

def dcg_at_k(rel, k):
    return sum(rel[i] / math.log2(i + 2) for i in range(min(k, len(rel))))

def ndcg_at_k(rel, k):
    ideal = sorted(rel, reverse=True)
    ideal_dcg = dcg_at_k(ideal, k)
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(rel, k) / ideal_dcg


In [ ]:
def evaluate_ndcg_gold(gold_df, k=5):
    scores = []

    for _, row in gold_df.iterrows():
        query = row["gold_query"]
        issue_id = row["issue_id"]

        retrieved = retrieve_indices(query, k=k)

        relevance = [
            1 if metadata[idx]["issue_id"] == issue_id else 0
            for idx in retrieved
        ]

        scores.append(ndcg_at_k(relevance, k))

    return np.mean(scores)


### Recall@K

In [ ]:
def evaluate_recall_gold(gold_df, k=10):
    hits = 0

    for _, row in gold_df.iterrows():
        query = row["gold_query"]
        issue_id = row["issue_id"]

        retrieved = retrieve_indices(query, k=k)

        if any(metadata[idx]["issue_id"] == issue_id for idx in retrieved):
            hits += 1

    return hits / len(gold_df)


### Evaluasi

In [ ]:
#acc_g = evaluate_accuracy_gold(gold_df)
#mrr_g = evaluate_mrr_gold(gold_df)
#ndcg_g = evaluate_ndcg_gold(gold_df)
#recall_5  = evaluate_recall_gold(gold_df, k=5)
#recall_10 = evaluate_recall_gold(gold_df, k=10)

#print("=== GOLD QUERY SET EVALUATION ===")
#print(f"Accuracy@1 : {acc_g:.4f}")
#print(f"MRR@10     : {mrr_g:.4f}")
#print(f"nDCG@5     : {ndcg_g:.4f}")
#print("Recall@5  :", f"{recall_5:.4f}")
#print("Recall@10 :", f"{recall_10:.4f}")


In [ ]:
#aaa a a dada dadad

# 11. GENERATION MODEL

In [ ]:
!pip install -q transformers accelerate sentencepiece
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch


## 11.1 Load FLAN-T5

In [ ]:
MODEL_NAME = "google/flan-t5-large"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
generator = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

generator.eval()


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
       

## 11.2 Context Construction

In [ ]:
ISSUE_TEMPLATE_LINES = {
    "describe the problem.",
    "steps to reproduce",
    "expected behavior",
    "actual behavior",
    "environment",
    "screenshots",
}


In [ ]:
MAX_CHUNKS_PER_ISSUE = 10
MAX_ISSUES = 3

In [ ]:
def select_top_issues_from_chunks(chunk_idxs, max_issues=3):
    seen = set()
    issue_ids = []

    for idx in chunk_idxs:
        issue_id = metadata[idx]["issue_id"]
        if issue_id not in seen:
            seen.add(issue_id)
            issue_ids.append(issue_id)
        if len(issue_ids) >= max_issues:
            break

    return issue_ids

In [ ]:
from collections import defaultdict

def group_chunks_by_issue(chunk_idxs):
    issues = defaultdict(list)

    for idx in chunk_idxs:
        issue_id = metadata[idx]["issue_id"]


        if len(issues[issue_id]) < MAX_CHUNKS_PER_ISSUE:
            issues[issue_id].append(idx)


    for issue_id in issues:
        issues[issue_id].sort(
            key=lambda i: metadata[i]["type"] != "issue_answer"
        )

    return issues

In [ ]:
def clean_context(text):
    text = BeautifulSoup(text, "html.parser").get_text()

    text = re.sub(r'!\[.*?\]\(.*?\)', ' ', text)
    text = re.sub(r'<img.*?>', ' ', text)
    text = re.sub(r'http\S+', ' ', text)

    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
def build_issue_representation(issue_id, trigger_chunk_idx,
                               max_chunks=10,
                               max_body_chars=600):
    texts = []


    for i, meta in enumerate(metadata):
        if meta["issue_id"] == issue_id and meta["type"] == "issue_body":
            body = clean_context(corpus[i])[:max_body_chars]
            texts.append(f"Issue description:\n{body}")
            break


    texts.append(f"Relevant discussion:\n{clean_context(corpus[trigger_chunk_idx])}")


    added = 1
    for i, meta in enumerate(metadata):
        if meta["issue_id"] != issue_id:
            continue
        if meta["type"] != "issue_answer":
            continue
        if i == trigger_chunk_idx:
            continue

        chunk = clean_context(corpus[i])
        if len(chunk.split()) < 15:
            continue

        texts.append(f"Additional discussion:\n{chunk}")
        added += 1
        if added >= max_chunks:
            break

    return "\n\n".join(texts)

In [ ]:
def is_template_line(line: str) -> bool:
    l = line.strip().lower()

    TEMPLATE_LINES = {
        "describe the problem.",
        "describe the problem",
        "steps to reproduce:",
        "expected behavior:",
        "actual behavior:",
        "additional context:",
        "environment:",
        "reproduction steps:",
        "expected result:",
        "actual result:",
    }

    return l in TEMPLATE_LINES

In [ ]:
def build_full_issue_context(
    issue_id,
    max_body_chars=900,
    max_answer_chunks=5
):
    parts = []


    for i, meta in enumerate(metadata):
        if meta["issue_id"] == issue_id and meta["type"] == "issue_body":
            raw = clean_context(corpus[i])

            lines = [
                l for l in raw.split("\n")
                if len(l.strip()) > 5
                and not is_template_line(l)
            ]

            text = " ".join(lines)[:max_body_chars]
            if text.strip():
                parts.append("Issue description:\n" + text)
            break


    discussion_chunks = []

    for i, meta in enumerate(metadata):
        if meta["issue_id"] != issue_id:
            continue
        if meta["type"] != "issue_answer":
            continue

        text = clean_context(corpus[i])
        if len(text.split()) < 8:
            continue

        discussion_chunks.append(text)


    for c in discussion_chunks[:max_answer_chunks]:
        parts.append("Relevant discussion:\n" + c)


    if not parts:
        parts.append(
            "The issue report contains very limited information and follows a template format."
        )

    return "\n\n".join(parts)

In [ ]:
def extract_technical_terms(issue_id, max_terms=6):
    terms = set()

    for i, meta in enumerate(metadata):
        if meta["issue_id"] != issue_id:
            continue

        text = corpus[i].lower()
        for w in text.split():
            if (
                len(w) > 4
                and w.isalpha()
                and w not in {"github", "issue", "problem", "error", "bug"}
            ):
                terms.add(w)

        if len(terms) >= max_terms:
            break

    return list(terms)[:max_terms]

In [ ]:
def has_technical_signal(text):
    keywords = [
        "error", "exception", "failed", "cannot", "crash",
        "dependency", "build", "compile", "runtime",
        "permission", "timeout", "configuration"
    ]
    t = text.lower()
    return sum(k in t for k in keywords) >= 2

## 11.3 Prompt Engineering

In [ ]:
def compress_for_t5(issue_title, issue_context, max_sentences=4):
    sentences = nltk.sent_tokenize(issue_context)

    technical_keywords = [
        "error", "exception", "failed",
        "cannot", "timeout", "403",
        "session", "crash", "stack",
        "websocket", "socket", "connection",
        "windows", "linux", "macos"
    ]


    bad_patterns = [
        "wslpeer",
        "examples",
        "tru)",
        "#",
        "set_nodelay",
        "i'm not sure",
        "not sure what",
        "could be the cause"
    ]

    speculative_patterns = [
        "might be related",
        "possible that",
        "could be caused",
        "not sure",
        "i'm not sure",
        "not sure what could",
        "might be the cause",
        "could be the cause",
        "some re-render",
        "misconfiguration"
    ]

    chatty_patterns = [
        "please post it here",
        "if anyone facing",
        "thank you",
        "i'm not sure",
        "can anyone help",
        "please help"
    ]

    template_noise = [
        "steps to reproduce",
        "to reproduce",
        "go to",
        "run simplified",
        "click log in"
    ]

    scored = []

    title_words = set(issue_title.lower().split())

    for s in sentences:
        s_clean = s.strip()
        s_lower = s_clean.lower()

        if any(p in s_lower for p in speculative_patterns):
          continue

        if any(p in s_lower for p in template_noise):
          continue

        if any(p in s_lower for p in chatty_patterns):
          continue

        if len(s_clean.split()) < 8:
            continue

        if any(p in s_lower for p in bad_patterns):
            continue

        overlap = len(
            title_words & set(s_lower.split())
        ) / max(len(title_words), 1)

        if overlap > 0.6:
            continue

        score = sum(k in s_lower for k in technical_keywords)

        if any(k in s_lower for k in [
            "windows", "macos", "connection",
            "error", "fails", "reset"
        ]):
            score += 2

        scored.append((score, s_clean))

    scored.sort(key=lambda x: x[0], reverse=True)

    selected = [s for _, s in scored[:max_sentences]]

    if selected:
        return " ".join(selected)

    return issue_context[:400]

In [ ]:
def build_issue_prompt(compressed_context):
    return f"""
Context:
{compressed_context}

Factual technical summary:
""".strip()

## 11.4 Summary Generation

In [ ]:
def infer_title_from_url(issue_id):
    for meta in metadata:
        if meta["issue_id"] == issue_id:
            url = meta["url"]
            repo = url.split("/")[-4]
            return f"Issue reported in repository '{repo}'"
    return None

In [ ]:
def summarize_issue_only(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(generator.device)

    with torch.no_grad():
        outputs = generator.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            length_penalty=1.1
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
def clean_generated_summary(summary):
    bad_suffixes = [
        "i'm not sure what could be the cause",
        "not sure what could be the cause",
        "could be the cause",
        "i'm not sure",
        "not sure"
    ]

    cleaned = summary.strip()

    for phrase in bad_suffixes:
        cleaned = re.sub(
            re.escape(phrase),
            "",
            cleaned,
            flags=re.IGNORECASE
        )

    cleaned = re.sub(r"\s+", " ", cleaned).strip(" .:-")

    return cleaned

# 12. END-TO-END RAG PIPELINE

## 12.1 Query Understanding

In [ ]:
def has_strong_technical_signal(text):
    keywords = [
        "error", "failed", "exception", "stack", "trace",
        "dependency", "build", "compile", "runtime",
        "permission", "timeout", "latency", "slow",
        "docker", "container", "android", "gradle"
    ]
    t = text.lower()
    return sum(k in t for k in keywords) >= 2

In [ ]:
def detect_query_intent(query):
    q = query.lower()

    infra = [
        "docker", "container", "volume", "mount",
        "permission", "filesystem", "unix"
    ]

    network = [
        "socket", "websocket", "http", "upgrade",
        "connection", "403", "reset"
    ]

    ui = [
        "ui", "interface", "frontend", "button",
        "layout", "responsive", "screen"
    ]

    infra_score = sum(k in q for k in infra)
    network_score = sum(k in q for k in network)
    ui_score = sum(k in q for k in ui)

    if infra_score > 0 and network_score == 0 and ui_score == 0:
        return "infra"
    if network_score > 0 and infra_score == 0 and ui_score == 0:
        return "network"
    if ui_score > 0 and infra_score == 0 and network_score == 0:
        return "ui"

    return "mixed"

In [ ]:
def is_query_relevant(query, chunk_idxs, min_overlap=1):
    if not chunk_idxs:
        return False

    top_idx = chunk_idxs[0]
    top_text = corpus[top_idx].lower()
    q_words = set(query.lower().split())

    overlap = len(q_words & set(top_text.split()))

    return overlap >= min_overlap

## 12.2 Issue Selection

In [ ]:
from collections import Counter

def select_top_issues_with_support(chunk_idxs, min_chunks=2, max_issues=3):
    counts = Counter(metadata[i]["issue_id"] for i in chunk_idxs)

    ranked = sorted(
        counts.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        issue_id
        for issue_id, cnt in ranked
        if cnt >= min_chunks
    ][:max_issues]

In [ ]:
def get_issue_title(issue_id):
    row = df[df["id"] == issue_id]

    if not row.empty:
        return str(row.iloc[0]["title"]).strip()

    return "Untitled issue"

## 12.3 Final Answer Pipeline

In [ ]:
def answer_query_issue_level(query, max_issues=3):
    chunk_idxs = retrieve_dense_reranked(query, k=80)

    if not is_query_relevant(query, chunk_idxs):
      return [{
          "issue_id": None,
          "title": "No relevant issue found",
          "summary": "The query does not match any issue in the current knowledge base.",
          "references": []
      }]

    intent = detect_query_intent(query)

    issue_ids = select_top_issues_with_support(
        chunk_idxs,
        max_issues=max_issues
    )

    results = []

    for issue_id in issue_ids:
        issue_title = get_issue_title(issue_id)
        issue_context = build_full_issue_context(issue_id)
        context_lower = issue_context.lower()

        if intent == "infra":
            if not any(k in context_lower for k in [
                "docker", "container", "volume", "mount", "unix"
            ]):
                continue

        if intent == "network":
            if not any(k in context_lower for k in [
                "socket", "websocket", "http", "upgrade", "connection"
            ]):
                continue

        if intent == "ui":
            if not any(k in context_lower for k in [
                "ui", "interface", "frontend", "button", "layout", "responsive", "screen"
            ]):
                continue

        if len(issue_context.split()) < 40:
            continue

        if not has_strong_technical_signal(issue_context) and intent == "mixed":
            title = ""
            for i, meta in enumerate(metadata):
                if meta["issue_id"] == issue_id and meta["type"] == "issue_body":
                    title = clean_context(corpus[i]).split(".")[0]
                    break

            summary = f"The issue reports: {issue_title}."

        else:
            issue_title = get_issue_title(issue_id)
            compressed_context = compress_for_t5(issue_title, issue_context)
            prompt = build_issue_prompt(compressed_context)
            summary = summarize_issue_only(prompt)
            summary = clean_generated_summary(summary)

        results.append({
            "issue_id": issue_id,
            "title": issue_title,
            "summary": summary,
            "references": list({
                meta["url"]
                for meta in metadata
                if meta["issue_id"] == issue_id
            })
        })

    return results

In [ ]:
for r in answer_query_issue_level("web socket error"):
    print("Title   :", r["title"])
    print("Summary :", r["summary"])
    print("Refs    :", r["references"])
    print("-"*60)

Title   : [WebSocket] "Unable to set TCP no delay option" error on Windows
Summary : This is a problem with the socket stack in windows
Refs    : ['https://github.com/godotengine/godot/issues/95217']
------------------------------------------------------------
Title   : (Starscream.HTTPUpgradeError.notAnUpgrade(403))
Summary : Using native engine websocket connection, authorization is working fine and then getting disconnected automatically
Refs    : ['https://github.com/daltoniam/Starscream/issues/966']
------------------------------------------------------------
Title   : Websocket throws "Unknown Session ID" when trying to connect
Summary : of the error: some re-render behaviour, our auth session logic or something misconfiguration of Socket.io, but in the end it succeeds
Refs    : ['https://github.com/wasp-lang/wasp/issues/2040']
------------------------------------------------------------


# 13. USER INTERFACE (GRADIO)

In [ ]:
import gradio as gr
import time

def web_answer(query):
    t0 = time.time()

    try:
        results = answer_query_issue_level(query)

        runtime = f"{time.time() - t0:.2f}s"

        if not results:
            return (
                "## ❌ No relevant technical issue found.",
                runtime
            )

        blocks = []

        for i, r in enumerate(results, 1):
            refs = "\n".join([f"- [{u}]({u})" for u in r["references"]])

            title = r.get("title", f"Issue {i}")
            summary = r.get("summary", "No summary available.")

            block = f"""
## 🔹 Issue {i}

### 🏷️ Title
{title}

### 📝 Summary
{summary}

### 🔗 References
{refs}
"""
            blocks.append(block)

        final_text = "\n\n---\n\n".join(blocks)

        return final_text, runtime

    except Exception as e:
        return f"## ❌ ERROR\n\n{str(e)}", "FAILED"


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
# 🧠 GitHub Issue RAG Chatbot
Semantic Search + SBERT + Reranker + Extractive Summary
""")

    with gr.Row():
        query_input = gr.Textbox(
            lines=2,
            placeholder="e.g. web socket error",
            label="Ask about GitHub issues"
        )
    gr.Examples(
        examples=[
            ["http 403 websocket issue"],
            ["connection reset by peer"],
            ["windows socket stack problem"]
        ],
        inputs=query_input
    )

    submit_btn = gr.Button("🔍 Search")

    output_md = gr.Markdown(label="Issue Results")
    runtime_box = gr.Textbox(label="⏱ Runtime")

    submit_btn.click(
        fn=web_answer,
        inputs=query_input,
        outputs=[output_md, runtime_box]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_693/3835921380.py:48: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://91dccb86c7c6a3efc4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://91dccb86c7c6a3efc4.gradio.live
